## 1. Import Libraries




In [21]:
import pandas as pd
import numpy as np
from scipy import stats
from sqlalchemy import create_engine
import glob
import os
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint
import statsmodels.api as sm

## 2. Load and Prepare Dataset

In [15]:
engine = create_engine(
    "mysql+pymysql://root:@localhost/air_quality"
)

try:
    with engine.connect():
        print("Connected to air_quality!")

    df = pd.read_sql_query("SELECT * FROM all_regions", engine)
    combined_df = df.copy()
    print(f"Loaded {len(df):,} records from all_regions.")
except Exception as error:
    raise RuntimeError(
        "Could not connect to MySQL or load the all_regions table. "
        "Check that MySQL is running and the air_quality database exists."
    ) from error

Connected to air_quality!
Loaded 420,768 records from all_regions.


## 3. Descriptive Statistics

Calculate mean, median, standard deviation, variance, min, max, Q1, Q3, and IQR for each pollutant. Describe the overall numerical characteristics of the air-quality data.

In [16]:
pollutants = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]

descriptive_stats = df[pollutants].agg([
    "count",
    "mean",
    "median",
    "std",
    "var",
    "min",
    "max"
])

# Calculate Q1 and Q3
q1 = df[pollutants].quantile(0.25)
q3 = df[pollutants].quantile(0.75)

# Add Q1 and Q3 to the table
descriptive_stats.loc["Q1"] = q1
descriptive_stats.loc["Q3"] = q3

# Calculate IQR
descriptive_stats.loc["IQR"] = q3 - q1

# Transpose the table so pollutants are rows
descriptive_stats = descriptive_stats.T

# Display the results
descriptive_stats

,count,mean,median,std,var,min,max,Q1,Q3,IQR
PM2.5,412029.0,79.793428,55.0,80.822391,6.532259e+03,2.0000,999.0,20.0,111.0,91.0
PM10,414319.0,104.602618,82.0,91.772426,8.422178e+03,2.0000,999.0,36.0,145.0,109.0
SO2,411747.0,15.830835,7.0,21.650603,4.687486e+02,0.2856,500.0,3.0,20.0,17.0
NO2,408652.0,50.638586,43.0,35.127912,1.233970e+03,1.0265,290.0,23.0,71.0,48.0
CO,400067.0,1230.766454,900.0,1160.182716,1.346024e+06,100.0000,10000.0,500.0,1500.0,1000.0
O3,407491.0,57.372271,45.0,56.661607,3.210538e+03,0.2142,1071.0,11.0,82.0,71.0


In [24]:
confidence_results = []

for pollutant in pollutants:

    # Remove missing values
    values = df[pollutant].dropna()

    # Sample size
    n = len(values)

    # Mean
    mean = values.mean()

    # Standard error
    standard_error = values.std() / np.sqrt(n)

    # 95% confidence interval using statsmodels
    ci_low, ci_high = sm.stats.DescrStatsW(values).tconfint_mean(
        alpha=0.05
    )

    confidence_results.append({
        "Pollutant": pollutant,
        "Mean": mean,
        "95% CI Lower": ci_low,
        "95% CI Upper": ci_high
    })

confidence_df = pd.DataFrame(confidence_results)

confidence_df

,Pollutant,Mean,95% CI Lower,95% CI Upper
0,PM2.5,79.793428,79.546644,80.040212
1,PM10,104.602618,104.323174,104.882061
2,SO2,15.830835,15.764704,15.896966
3,NO2,50.638586,50.530883,50.746288
4,CO,1230.766454,1227.171367,1234.361541
5,O3,57.372271,57.198299,57.546243


## 4. Station-Level Analysis 


In [25]:
# ============================================
# 2. STATION-LEVEL ANALYSIS
# ============================================

station_stats = df.groupby("station")[pollutants].agg([
    "mean",
    "median",
    "std"
])

station_stats

PM2.5                          PM10                    \
                    mean median        std        mean median        std   
station                                                                    
Aotizhongxin   82.773611   58.0  82.135694  110.060391   87.0  95.223005   
Changping      71.099743   46.0  72.326926   94.657871   72.0  83.441738   
Dingling       65.989497   41.0  72.267723   83.739723   60.0  79.541685   
Dongsi         86.194297   61.0  86.575127  110.336742   86.0  98.219860   
Guanyuan       82.933372   59.0  80.933497  109.023303   89.0  91.573709   
Gucheng        83.852089   60.0  82.796445  118.861978   99.0  96.742626   
Huairou        69.626367   47.0  71.224916   91.482690   69.0  83.289578   
Nongzhanguan   84.838483   59.0  86.225344  108.991096   85.0  95.341177   
Shunyi         79.491602   55.0  81.231739   98.737026   77.0  89.143718   
Tiantan        82.164911   59.0  80.921384  106.363672   85.0  89.700157   
Wanliu         83.374716   59.0  81.905568  110.464618   88.0  92.795065   
Wanshouxigong  85.024136   60.0  85.975981  112.223459   91.0  97.593210   

                     SO2                          NO2                    \
                    mean median        std       mean median        std   
station                                                                   
Aotizhongxin   17.375901    9.0  22.823017  59.305833   53.0  37.116200   
Changping      14.958906    7.0  20.975331  44.182086   36.0  29.519796   
Dingling       11.749650    5.0  15.519259  27.585467   19.0  26.383882   
Dongsi         18.531107   10.0  22.905655  53.699443   47.0  33.959230   
Guanyuan       17.590941    8.0  23.600367  57.901643   51.0  35.150857   
Gucheng        15.366162    7.0  21.204526  55.871075   50.0  36.473860   
Huairou        12.121553    4.0  18.896912  32.497250   25.0  26.489531   
Nongzhanguan   18.689242    9.0  24.280665  58.097172   51.0  36.297740   
Shunyi         13.572039    5.0  19.572068  43.908865   37.0  30.996828   
Tiantan        14.367615    7.0  20.144631  53.162646   47.0  31.946224   
Wanliu         18.376481   10.0  22.609648  65.258789   60.0  37.996088   
Wanshouxigong  17.148603    8.0  23.940834  55.529560   49.0  35.808050   

                        CO                              O3                      
                      mean  median          std       mean   median        std  
station                                                                         
Aotizhongxin   1262.945145   900.0  1221.436236  56.353358  42.0000  57.916327  
Changping      1152.301345   800.0  1103.056282  57.940003  46.0000  54.316674  
Dingling        904.896073   600.0   903.306220  68.548371  61.0000  53.764424  
Dongsi         1330.069131  1000.0  1191.305887  57.210637  44.1252  58.033275  
Guanyuan       1271.294377   900.0  1164.854945  55.795044  41.0000  57.436983  
Gucheng        1323.974423   900.0  1208.957772  57.694879  45.0000  57.019587  
Huairou        1022.554545   800.0   898.738241  59.824713  49.0000  54.605746  
Nongzhanguan   1324.350198   900.0  1245.166124  58.534682  45.0000  58.401448  
Shunyi         1187.063979   800.0  1156.374102  55.201321  43.0000  54.873726  
Tiantan        1298.303318   900.0  1170.593297  55.984297  40.0000  59.081528  
Wanliu         1319.353513   900.0  1268.114331  48.873614  32.0000  55.111740  
Wanshouxigong  1370.395031  1000.0  1223.139114  56.229904  42.0000  57.082710

In [26]:
# Create a station-level summary table
station_summary = []

for station, group in df.groupby("station"):

    for pollutant in pollutants:

        values = group[pollutant].dropna()

        station_summary.append({
            "Station": station,
            "Pollutant": pollutant,
            "Mean": values.mean(),
            "Median": values.median(),
            "Standard Deviation": values.std()
        })

station_summary_df = pd.DataFrame(station_summary)

station_summary_df

,Station,Pollutant,Mean,Median,Standard Deviation
0,Aotizhongxin,PM2.5,82.773611,58.0,82.135694
1,Aotizhongxin,PM10,110.060391,87.0,95.223005
2,Aotizhongxin,SO2,17.375901,9.0,22.823017
3,Aotizhongxin,NO2,59.305833,53.0,37.116200
4,Aotizhongxin,CO,1262.945145,900.0,1221.436236
...,...,...,...,...,...
67,Wanshouxigong,PM10,112.223459,91.0,97.593210
68,Wanshouxigong,SO2,17.148603,8.0,23.940834
69,Wanshouxigong,NO2,55.529560,49.0,35.808050
70,Wanshouxigong,CO,1370.395031,1000.0,1223.139114


In [27]:
# ============================================
# HIGHEST MEAN CONCENTRATION BY STATION
# ============================================

highest_mean_results = []

for pollutant in pollutants:

    station_means = df.groupby("station")[pollutant].mean()

    highest_station = station_means.idxmax()
    highest_mean = station_means.max()

    highest_mean_results.append({
        "Pollutant": pollutant,
        "Station with Highest Mean": highest_station,
        "Mean Concentration": highest_mean
    })

highest_mean_df = pd.DataFrame(highest_mean_results)

highest_mean_df


,Pollutant,Station with Highest Mean,Mean Concentration
0,PM2.5,Dongsi,86.194297
1,PM10,Gucheng,118.861978
2,SO2,Nongzhanguan,18.689242
3,NO2,Wanliu,65.258789
4,CO,Wanshouxigong,1370.395031
5,O3,Dingling,68.548371


In [ ]:
# ============================================
# STATION-LEVEL 95% CONFIDENCE INTERVAL
# ============================================

station_confidence = []

for station, group in df.groupby("station"):

    for pollutant in pollutants:

        values = group[pollutant].dropna()

        if len(values) > 1:

            # Calculate 95% confidence interval using statsmodels
            ci_low, ci_high = sm.stats.DescrStatsW(
                values
            ).tconfint_mean(alpha=0.05)

            station_confidence.append({
                "Station": station,
                "Pollutant": pollutant,
                "Mean": values.mean(),
                "95% CI Lower": ci_low,
                "95% CI Upper": ci_high
            })

station_confidence_df = pd.DataFrame(station_confidence)

pd.set_option("display.max_rows", None)

,Station,Pollutant,Mean,95% CI Lower,95% CI Upper
0,Aotizhongxin,PM2.5,82.773611,81.902306,83.644915
1,Aotizhongxin,PM10,110.060391,109.053304,111.067479
2,Aotizhongxin,SO2,17.375901,17.133757,17.618046
3,Aotizhongxin,NO2,59.305833,58.911534,59.700132
4,Aotizhongxin,CO,1262.945145,1249.823423,1276.066867
...,...,...,...,...,...
67,Wanshouxigong,PM10,112.223459,111.194802,113.252115
68,Wanshouxigong,SO2,17.148603,16.895583,17.401623
69,Wanshouxigong,NO2,55.529560,55.150652,55.908468
70,Wanshouxigong,CO,1370.395031,1357.348553,1383.441508
